# NDT cells without ground removal: how many points do the cells cover?

One question only: what changes if the ground is **not** removed before G builds the 2000 NDT cells.
For a random sample of scans the cells are built twice from the same cropped scan (`scripts/ground_coverage.py`):

- `patchwork`: ground removed with Patchwork++ first (the project default);
- `none`: every cropped point, ground included.

Patchwork++ labels every cropped point as ground / non-ground, and that label is used for both modes. Per scan and mode:

- `cov_all`, `cov_nonground`, `cov_ground`: % of the points within 0.8 m (the cell radius) of a cell mean;
- `cells_ground`: % of the 2000 cells spent on the ground;
- `voxel`: voxel size found by G.

Nothing is trained, and no GPU is needed. The results go to `MyDrive/lpr/ground_check/<hash>/`, together with the parameters.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/dov85/lidar-place-recognition"
DRIVE = "/content/drive/MyDrive/lpr"
KITTI = "/content/kitti"
SEQS = ["00"]          # raw scans: MyDrive/lpr/kitti_seq<seq>_raw.tar if there, otherwise downloaded (08: a few GB)
N_SCANS = 300          # random scans per sequence (0 = all); about 1-2 s per scan for both modes on 2 cores
PREVIEWS = 6           # scans with a picture
NDT_PARAMS = dict()    # only what differs from NDT_DEFAULTS (e.g. dict(beams=16)); ground is set by the script

## 2. Drive, code

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, glob, subprocess, importlib
url = REPO_URL
try:  # private repo: store a token in Colab Secrets as GITHUB_TOKEN
    url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass
if not os.path.exists("/content/lpr"):
    subprocess.run(["git", "clone", "-q", url, "/content/lpr"], check=True)
else:
    subprocess.run(["git", "-C", "/content/lpr", "pull", "-q"], check=True)
%cd /content/lpr
!git log --oneline -1
!pip install -q -r requirements.txt
sys.path.insert(0, "/content/lpr/src")
import lpr.experiments as ex
importlib.reload(ex)

FULL = {**ex.NDT_DEFAULTS, **NDT_PARAMS}
FULL.pop("ground")                       # both modes are run
TAG = ex.ndt_hash(dict(FULL, n_scans=N_SCANS))
OUT = f"{DRIVE}/ground_check/{TAG}"
os.makedirs(OUT, exist_ok=True)
print("results:", OUT)
print(json.dumps(FULL))

## 3. Raw scans
In this order: already on this machine; `MyDrive/lpr/kitti_seq<seq>_raw.tar` (if it is not empty); the Azure blob container
(`raw/kitti_seq<seq>_raw.tar`, built by `azure_build.sh` for 00 02 05 06 08; SAS URL in Colab Secrets as `AZURE_SAS`);
otherwise downloaded from KITTI.

In [ ]:
import requests, shutil

def from_blob(seq, dest):
    """raw/kitti_seq<seq>_raw.tar from the blob container to dest; False if not there."""
    try:
        sas = userdata.get("AZURE_SAS")
    except Exception:
        return False
    base, token = sas.split("?", 1)
    with requests.get(f"{base.rstrip('/')}/raw/kitti_seq{seq}_raw.tar?{token}", stream=True, timeout=60) as r:
        if r.status_code != 200:
            print(f"{seq}: not in the blob container (HTTP {r.status_code})")
            return False
        with open(dest, "wb") as f:
            shutil.copyfileobj(r.raw, f, length=16 << 20)
    return True

os.makedirs(KITTI, exist_ok=True)
for s in SEQS:
    vel = f"{KITTI}/sequences/{s}/velodyne"
    if os.path.isdir(vel) and os.listdir(vel):
        print(s, "already here")
        continue
    raw = f"{DRIVE}/kitti_seq{s}_raw.tar"
    tmp = f"/content/raw_{s}.tar"
    if os.path.exists(raw) and os.path.getsize(raw) > 0:
        print(s, "from Drive")
        !tar -xf {raw} -C {KITTI}
    elif from_blob(s, tmp):
        print(s, "from the blob container")
        !tar -xf {tmp} -C {KITTI} && rm -f {tmp}
    else:
        print(s, "downloading from KITTI")
        !python scripts/download_kitti.py --out {KITTI} --seqs {s} --threads 48
    assert os.path.isdir(vel) and os.listdir(vel), f"no raw scans of sequence {s} in {vel}"
    print(s, len(os.listdir(vel)), "scans")


## 4. Cells with and without ground removal
A sequence whose results are already in `OUT` is not run again.

In [ ]:
params = json.dumps({k: v for k, v in NDT_PARAMS.items() if k != "ground"})
for s in SEQS:
    if os.path.exists(f"{OUT}/ground_coverage_{s}_summary.csv"):
        print(s, "done before")
        continue
    !python scripts/ground_coverage.py --kitti_root {KITTI} --seq {s} --n_scans {N_SCANS} --previews {PREVIEWS} --out_dir {OUT} --params '{params}'

## 5. Results
Means over the scans. The question is the `cov_all` row (all cropped points, ground included) and what happens to `cov_nonground`
(the structures) when part of the 2000 cells goes to the ground (`cells_ground`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
S = pd.concat([pd.read_csv(f"{OUT}/ground_coverage_{s}_summary.csv", dtype={"seq": str}) for s in SEQS
               if os.path.exists(f"{OUT}/ground_coverage_{s}_summary.csv")])
display(S.set_index(["seq", "mode"]).T)
S.to_csv(f"{OUT}/summary_all.csv", index=False)

D = pd.concat([pd.read_csv(f"{OUT}/ground_coverage_{s}.csv", dtype={"seq": str}) for s in SEQS
               if os.path.exists(f"{OUT}/ground_coverage_{s}.csv")])
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col in zip(axes, ["cov_all", "cov_nonground", "cov_ground", "cells_ground"]):
    for mode, c in [("patchwork", "tab:blue"), ("none", "tab:red")]:
        ax.hist(D[D["mode"] == mode][col], bins=30, alpha=0.6, color=c,
                label="ground removed" if mode == "patchwork" else "ground kept")
    ax.set_title(col + " (%)")
    ax.legend()
fig.tight_layout()
fig.savefig(f"{OUT}/hist.png", dpi=100)
plt.show()

# per scan: change in non-ground coverage when the ground is kept
W = D.pivot_table(index=["seq", "idx"], columns="mode", values=["cov_all", "cov_nonground"])
delta = (W["cov_nonground"]["none"] - W["cov_nonground"]["patchwork"])
print(f"non-ground coverage, ground kept minus removed: mean {delta.mean():+.2f} points of %, "
      f"5th pct {delta.quantile(0.05):+.2f}, scans worse by > 5: {(delta < -5).mean() * 100:.1f}%")

## 6. Pictures
Left: the cropped scan (olive: ground). Middle: cells with ground removal. Right: cells without.

In [ ]:
from IPython.display import Image, display
for s in SEQS:
    for png in sorted(glob.glob(f"{OUT}/ground_previews_{s}/*.png")):
        print(s, os.path.basename(png))
        display(Image(png, width=1400))